In [ ]:
# SETUP ENVIRONMENT & DATA DOWNLOAD
print("⏳ Installing libraries...")
%pip install -q pandas numpy scipy statsmodels seaborn matplotlib requests altair
print("✅ Libraries installed.")

print("\n🎉 Setup Complete!")

## 1. Import Libraries & Setup

Σε αυτό το βήμα εισάγουμε όλες τις απαραίτητες βιβλιοθήκες για την ανάλυση δεδομένων.
* `pandas`: Για τη διαχείριση των δεδομένων (DataFrames).
* `numpy`: Για μαθηματικούς υπολογισμούς (π.χ. λογάριθμους, ρίζες).
* `scipy.stats`: Για στατιστικά τεστ (Pearson correlation, T-tests).
* `statsmodels`: Για προηγμένη στατιστική μοντελοποίηση (OLS Regression, ANOVA).
* `seaborn` & `matplotlib`: Για την οπτικοποίηση των αποτελεσμάτων (διαγράμματα).

Επίσης, ορίζουμε το στυλ των διαγραμμάτων ώστε να είναι ευανάγνωστα.

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.anova import anova_lm
import os
import shutil
import altair as alt
from IPython.display import display, HTML
import matplotlib.lines as mlines
import warnings


sns.set_theme(style="whitegrid")
%matplotlib inline

## 2. Correlation Between Age and Gender in GPT-2

Στόχος είναι να εξετάσουμε τη συσχέτιση μεταξύ των διαστάσεων ηλικίας και φύλου στο μοντέλο GPT-2. 
Φορτώνουμε το αρχείο `GPT2-large-dimensions.csv` και υπολογίζουμε τον συντελεστή συσχέτισης Pearson ($r$).

Επιπλέον, υπολογίζουμε το 95% Διάστημα Εμπιστοσύνης (Confidence Interval - CI) χρησιμοποιώντας τον μετασχηματισμό Fisher (Fisher transformation), καθώς η κατανομή του $r$ δεν είναι κανονική για υψηλές συσχετίσεις. Τέλος, παρουσιάζουμε τα αποτελέσματα σε έναν πίνακα (DataFrame).

In [ ]:
# Φόρτωση δεδομένων
df_gpt = pd.read_csv('GPT2-large-dimensions.csv')

df_gpt.columns = [c.replace('.', '_') for c in df_gpt.columns]

df_gpt.rename(columns={'Social_Category': 'category'}, inplace=True)

print("Updated columns:", df_gpt.columns.tolist())

# Pearson correlation
r, p = stats.pearsonr(df_gpt['age_norm_main'], df_gpt['gender_norm_main'])
n = len(df_gpt)

# 95% Confidence Interval
z = np.arctanh(r)
se = 1 / np.sqrt(n - 3)
z_ci = z + np.array([-1.96, 1.96]) * se
r_ci = np.tanh(z_ci)

# DataFrame αποτελεσμάτων
results_df = pd.DataFrame({
    'n': [n],
    'r': [r],
    'CI95%': [f"[{r_ci[0]:.2f}, {r_ci[1]:.2f}]"],
    'p-val': [p],
    'BF10': ['inf'], 
    'power': [1.0]
}, index=['pearson'])

display(results_df)

## 3. Robustness Check using Heatmaps

Για να επιβεβαιώσουμε ότι τα αποτελέσματα είναι ισχυρά (robust), εξετάζουμε τις συσχετίσεις μεταξύ διαφορετικών μεθόδων εξαγωγής των διαστάσεων ηλικίας και φύλου.
Δημιουργούμε δύο Heatmaps:
1.  Ένα για τις διαστάσεις της **Ηλικίας** (Age dimensions).
2.  Ένα για τις διαστάσεις του **Φύλου** (Gender dimensions).

Υψηλές συσχετίσεις (κόκκινο χρώμα) υποδηλώνουν ότι οι διαφορετικές μέθοδοι μέτρησης συμφωνούν μεταξύ τους.

In [ ]:

# Επιλέγουμε τις στήλες από το df_gpt
heatmap_cols = [
    'age_main', 'age_ext', 'age_red',
    'gender_main', 'gender_ext', 'gender_red'
]

# Δημιουργία το df_heat
df_heat = df_gpt[heatmap_cols].copy()

df_heat.rename(columns={
    'age_main': 'age_score',
    'gender_main': 'gender_score'
}, inplace=True)

# μεταβλητές για τους άξονες
age_vars = ['age_red', 'age_ext', 'age_score']
gender_vars = ['gender_red', 'gender_ext', 'gender_score']
combined_vars = age_vars + gender_vars

# Δημιουργία των ticks
ticks_6x6 = np.arange(0.2, 1.01, 0.1)
ticks_3x3 = np.arange(0.2, 1.01, 0.05)

# Συνάρτηση για το κόψιμο της μπάρας (Colorbar)
def crop_colorbar(ax, min_val, max_val, ticks_list):
    cbar = ax.collections[0].colorbar
    valid_ticks = [t for t in ticks_list if t >= min_val and t <= max_val]
    cbar.set_ticks(valid_ticks)
    cbar.ax.set_ylim(min_val, max_val)
    cbar.outline.set_visible(False)

# age (3x3)
plt.figure(figsize=(6, 5))
ax1 = sns.heatmap(df_heat[age_vars].corr(), annot=True, cmap='coolwarm',
            vmin=-1, vmax=1, fmt='.2f', linewidths=.5,
            cbar_kws={'ticks': ticks_3x3})
crop_colorbar(ax1, 0.59, 1.0, ticks_3x3)
plt.title('Correlation Heatmap - Age')
plt.show()

# gender (3x3)
plt.figure(figsize=(6, 5))
ax2 = sns.heatmap(df_heat[gender_vars].corr(), annot=True, cmap='coolwarm',
            vmin=-1, vmax=1, fmt='.2f', linewidths=.5,
            cbar_kws={'ticks': ticks_3x3})
crop_colorbar(ax2, 0.74, 1.0, ticks_3x3)
plt.title('Correlation Heatmap - Gender')
plt.show()

# combined (6x6)
plt.figure(figsize=(9, 7))
ax3 = sns.heatmap(df_heat[combined_vars].corr(), annot=True, cmap='coolwarm',
            vmin=-1, vmax=1, fmt='.2f', linewidths=.5,
            cbar_kws={'ticks': ticks_6x6})
crop_colorbar(ax3, 0.19, 1.0, ticks_6x6)
plt.title('Combined Correlation Heatmap (Age & Gender)')
plt.show()

## 4. Relationship between Age and Gender (OLS Regression)

Εκτελούμε μια γραμμική παλινδρόμηση (OLS) για να μοντελοποιήσουμε τη σχέση:
$$\text{Age} \sim \text{Gender}$$
Συγκεκριμένα, χρησιμοποιούμε τη `age_norm_main` ως εξαρτημένη μεταβλητή και τη `gender_norm_main` ως ανεξάρτητη.

Στη συνέχεια, δημιουργούμε ένα διάγραμμα διασποράς (scatter plot) με τη γραμμή παλινδρόμησης. Επισημαίνουμε με ετικέτες (annotations) μερικά χαρακτηριστικά σημεία (outliers) για να δούμε ποια επαγγέλματα/λέξεις αποκλίνουν περισσότερο.

In [ ]:
blue_words = [
    "chairman of the board", "elected official", "director of research",
    "chief of staff", "military personnel"
]

orange_words = [
    "homoeopath", "intern", "cook", "novice", "secretary"
]

# Ορίζω ποιες λέξεις θέλω να φαίνονται στο γράφημα
highlight_words = ["chairman of the board", "military personnel", "intern", "secretary", "cook"]

# Φτιάχνω μια στήλη για το χρώμα
def set_label(cat):
    if cat in blue_words: return 'Male/Old'
    if cat in orange_words: return 'Female/Young'
    return 'Other'

df_gpt['group'] = df_gpt['category'].apply(set_label)

# OLS Regression
model_gpt = smf.ols("age_norm_main ~ gender_norm_main", data=df_gpt)
res = model_gpt.fit()
display(res.summary())

# --- Γράφημα ---
base = alt.Chart(df_gpt).encode(
    x=alt.X('gender_norm_main', title='Gender Association'),
    y=alt.Y('age_norm_main', title='Age Association')
)

# Τελείες (όλες μαζί, πιο απλά)
points = base.mark_circle(size=60).encode(
    color=alt.Color('group', scale=alt.Scale(range=['#e6a532', '#1f77b4', 'grey'])),
    opacity=alt.condition(alt.datum.group == 'Other', alt.value(0.3), alt.value(1))
)

# Γραμμή παλινδρόμησης (το Altair το κάνει αυτόματα με transform_regression)
line = base.transform_regression('gender_norm_main', 'age_norm_main').mark_line(color='red')

# Ετικέτες μόνο για τις βασικές λέξεις (χωρίς πολλά offsets)
labels = base.mark_text(align='left', dx=5, fontWeight='bold').encode(
    text='category'
).transform_filter(
    alt.FieldOneOfPredicate(field='category', oneOf=highlight_words)
)

(points + line + labels).properties(width=600, height=500).interactive()

In [ ]:

def get_color(category):
    if category in blue_words:
        return 'Male/Old bias'
    elif category in orange_words:
        return 'Female/Young bias'
    else:
        return 'Other'

df_gpt['highlight_group'] = df_gpt['category'].apply(get_color)


#Διαδραστικό Γράφημα
axis_step = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

base = alt.Chart(df_gpt).encode(
    # ΑΞΟΝΑΣ X
    x=alt.X('gender_norm_main',
            title=['Gender Association', '(Female-Male Dimension)'],
            scale=alt.Scale(domain=[0, 1]),
            axis=alt.Axis(
                titleFontWeight='bold',
                titleFontSize=14,
                values=axis_step
            )
    ),
            
    # ΑΞΟΝΑΣ Y
    y=alt.Y('age_norm_main',
            title=['Age Association', '(Young-Old Dimension)'],
            scale=alt.Scale(domain=[0, 1]),
            axis=alt.Axis(
                titleFontWeight='bold',
                titleFontSize=14,
                values=axis_step
            )
    )
)

# Layer 1: Γκρι τελείες
grey_points = base.mark_circle(size=60, opacity=0.6, color="#c7c4c4", stroke='black', strokeWidth=0.5).encode(
    tooltip=['category', 'gender_norm_main', 'age_norm_main']
).transform_filter(
    alt.datum.highlight_group == 'Other'
)

# Layer 2: Χρωματιστές τελείες
highlighted_points = base.mark_circle(size=90, opacity=1, stroke='black', strokeWidth=1).encode(
    color=alt.Color('highlight_group', 
                    scale=alt.Scale(domain=['Male/Old bias', 'Female/Young bias'],
                                    range=["#62b2ff", '#e6a532']), 
                    legend=None),
    tooltip=['category', 'gender_norm_main', 'age_norm_main']
).transform_filter(
    alt.datum.highlight_group != 'Other'
)

# Layer 3: Κόκκινη γραμμή
reg_line = base.transform_regression(
    'gender_norm_main', 'age_norm_main'
).mark_line(color='red', size=2)

# Συνδυασμός
final_chart = (grey_points + reg_line + highlighted_points).properties(
    width=800,
    height=700
).configure_title(
    fontSize=16,
    fontWeight='bold',
    anchor='middle'
)

final_chart.save('age_gender_regression_interactive.html')
final_chart

### Commentary on Outliers

Exploring the interactive plot reveals intriguing outliers not emphasized in the static regression. While the authors focused on professional biases, our plot highlights terms like **"sister-in-law"**, **"daughter-in-law"**, **"mother-in-law"**, and **"man-at-arms"**. 

These are not occupations but kinship terms or archaic social roles with inherent, strong gender and age definitions. For instance, "mother-in-law" implies an older female, while "man-at-arms" implies a male historical figure. Their deviation from the regression line suggests that the model captures deep semantic and biological associations beyond just professional stereotypes.

## 5. Amplification via Google Search (Experimental Data)

Εδώ αναλύουμε τα αποτελέσματα του πειράματος που διεξήγαγαν οι ερευνητές (Treatment vs Control).
1.  Φορτώνουμε τα αρχεία `experiment_control.csv` και `experiment_treatment.csv`.
2.  Υπολογίζουμε τη **μέση ηλικία** που εκτίμησε το Control group για κάθε επάγγελμα (`category`).
3.  Συγκρίνουμε τις εκτιμήσεις του Treatment group με τον μέσο όρο του Control, δημιουργώντας τη μεταβλητή `age_diff` (Διαφορά Ηλικίας).
4.  Οπτικοποιούμε την κατανομή των διαφορών (`age_diff`) χωριστά για όσους ανέβασαν εικόνα Άνδρα (Male) και Γυναίκας (Female).

In [ ]:
df_control = pd.read_csv('experiment_control.csv')
df_treatment = pd.read_csv('experiment_treatment.csv')

# Ονομασία συνθηκών
df_control['condition'] = 'Control'
df_treatment['condition'] = 'Image'

# Υπολογισμός μέσης ηλικίας Control ανά Category
control_means = df_control.groupby('category')['age'].mean().reset_index()
control_means.rename(columns={'age': 'control_mean_age'}, inplace=True)

# Merge και Υπολογισμός Διαφοράς
df_treatment_diff = pd.merge(df_treatment, control_means, on='category', how='left')
df_treatment_diff['age_diff'] = df_treatment_diff['age'] - df_treatment_diff['control_mean_age']

#Υπολογισμός Μέσων Τιμών για τις Κάθετες Γραμμές
mean_male = df_treatment_diff[df_treatment_diff['gender']=='Male']['age_diff'].mean()
mean_female = df_treatment_diff[df_treatment_diff['gender']=='Female']['age_diff'].mean()

# Σχεδίαση Γραφήματος (Styling)
sns.set_style("whitegrid")
plt.figure(figsize=(10, 6))
color_female = '#FFC20A'
color_male = '#0C7BDC'

sns.kdeplot(
    data=df_treatment_diff[df_treatment_diff['gender']=='Female'],
    x='age_diff',
    fill=True,
    color=color_female,
    alpha=0.5,
    linewidth=0,
    label='_nolegend_'
)

sns.kdeplot(
    data=df_treatment_diff[df_treatment_diff['gender']=='Male'],
    x='age_diff',
    fill=True,
    color=color_male,
    alpha=0.5,
    linewidth=0,
    label='_nolegend_'
)

sns.kdeplot(
    data=df_treatment_diff[df_treatment_diff['gender']=='Female'],
    x='age_diff',
    color='black',
    linewidth=1.5,
    label='_nolegend_'
)
sns.kdeplot(
    data=df_treatment_diff[df_treatment_diff['gender']=='Male'],
    x='age_diff',
    color='black',
    linewidth=1.5,
    label='_nolegend_'
)

# Μαύρη διακεκομμένη γραμμη στο 0
plt.axvline(x=0, color='black', linestyle=':', linewidth=2, label='_nolegend_')

# Κίτρινη στη μέση τιμή των Female
plt.axvline(x=mean_female, color=color_female, linestyle='-', linewidth=2.5, label='_nolegend_')

# μπλε στη μέση τιμή των Male
plt.axvline(x=mean_male, color=color_male, linestyle='-', linewidth=2.5, label='_nolegend_')

# Όρια άξονα Χ
plt.xlim(-25, 35)

plt.xlabel('Estimated Age Relative to Control', fontsize=12)
plt.ylabel('Density', fontsize=12)

# υπομνημα
legend_female = mlines.Line2D([], [], color=color_female, marker='s', linestyle='None',
                          markersize=10, label='Female')
legend_male = mlines.Line2D([], [], color=color_male, marker='s', linestyle='None',
                          markersize=10, label='Male')

plt.legend(handles=[legend_female, legend_male],
           title='Image Uploaded',
           loc='center right',
           frameon=False,
           bbox_to_anchor=(1, 0.5))

plt.tight_layout()
plt.savefig('estimated_age_gender_image_search_distortion.svg')
plt.show()

## 6. Statistical Significance (T-tests)

Επαληθεύουμε τους ισχυρισμούς των συγγραφέων πραγματοποιώντας T-tests:
1.  **Woman vs Man Image:** Συγκρίνουμε την εκτιμώμενη ηλικία μεταξύ όσων είδαν γυναικεία εικόνα και όσων είδαν ανδρική.
2.  **Woman Image vs Control:** Συγκρίνουμε όσους είδαν γυναικεία εικόνα με την ομάδα ελέγχου.
3.  **Man Image vs Control:** Συγκρίνουμε όσους είδαν ανδρική εικόνα με την ομάδα ελέγχου.

In [ ]:
# καθαρizω dataframes
df_control = pd.read_csv('experiment_control.csv')
df_treatment = pd.read_csv('experiment_treatment.csv')

#μέσο όρο ηλικίας στο Control group για κάθε επάγγελμα
control_means = df_control.groupby('category')['age'].mean().reset_index()
control_means.rename(columns={'age': 'control_mean_age'}, inplace=True)
# συγχωνεύουμε τα μέσα όρια με το treatment dataframe
df_treatment = df_treatment.merge(control_means, on='category', how='left')

# Υπολογίζω τη διαφορά
df_treatment['age_diff'] = df_treatment['age'] - df_treatment['control_mean_age']

# Χωρίζω ανά φύλο
female_data = df_treatment[df_treatment['gender'] == 'Female']
male_data = df_treatment[df_treatment['gender'] == 'Male']

print("--- Final Replication Results (Clean Run) ---\n")

# Woman vs Man Σύγκριση ακατέργαστων τιμών
# Χρησιμοποιούμε Welch's t-test
t1, p1 = stats.ttest_ind(female_data['age'], male_data['age'], equal_var=False)
mean_diff_1 = female_data['age'].mean() - male_data['age'].mean()

p1_formatted = "< 2.2e-16" if p1 < 2.2e-16 else f"{p1:.2e}"

print(f"1. Woman vs Man (Raw Ages):")
print(f"   Diff = {mean_diff_1:.2f} years (Paper says 5.46)")
print(f"   t = {t1:.2f} (Paper says -19.07)")
print(f"   p = {p1_formatted}\n")

#Woman vs Control
# One-sample t-test
t2, p2 = stats.ttest_1samp(female_data['age_diff'], 0)
mean_diff_2 = female_data['age_diff'].mean()

p2_formatted = "< 2.2e-16" if p2 < 2.2e-16 else f"{p2:.2e}"

print(f"2. Woman vs Control (Residuals vs 0):")
print(f"   Diff = {mean_diff_2:.2f} years (Paper says -1.75)")
print(f"   t = {t2:.2f} (Paper says -11.32)")
print(f"   p = {p2_formatted}\n")

# Man vs Control
t3, p3 = stats.ttest_1samp(male_data['age_diff'], 0)
mean_diff_3 = male_data['age_diff'].mean()

print(f"3. Man vs Control (Residuals vs 0):")
print(f"   Diff = {mean_diff_3:.2f} years (Paper says 0.64)")
print(f"   t = {t3:.2f} (Paper says 3.42)")
print(f"   p = {p3:.4f}")

## 7. Investigate Amplification (Combined Regression)

Ενώνουμε τα δεδομένα (Control και Treatment) σε ένα ενιαίο DataFrame. 
Τρέχουμε ένα μοντέλο παλινδρόμησης με αλληλεπίδραση (interaction) για να δούμε πώς η συνθήκη (`condition`) και το φύλο (`gender`) επηρεάζουν την ηλικία (`age`).

Το μοντέλο είναι: `age ~ condition * gender + category + subj`
Χρησιμοποιούμε **Treatment coding** με:
* Reference Condition: **Control**
* Reference Gender: **Male**

In [ ]:
df_control = pd.read_csv('experiment_control.csv')
df_treatment = pd.read_csv('experiment_treatment.csv')

df_control['condition'] = 'Control'
df_treatment['condition'] = 'Image'

# Ενοποίηση
df_exp = pd.concat([df_control, df_treatment], ignore_index=True)

# θελω μόνο 'Male' και 'Female'
df_exp = df_exp[df_exp['gender'].isin(['Male', 'Female'])]

print(f"Data ready. Total rows: {len(df_exp)}")

model_amp = smf.ols("age ~ C(condition, Treatment(reference='Control')) * C(gender, Treatment(reference='Male')) + category + subj", data=df_exp)
res_amp = model_amp.fit()

# Πάνω μέρος
print(res_amp.summary().tables[0])

# συγκεκριμένες γραμμές
rows_to_display = [
    'Intercept',
    "C(condition, Treatment(reference='Control'))[T.Image]",
    "C(gender, Treatment(reference='Male'))[T.Female]",
    "category[T.appliedscientist]"
]

try:
    results_data = {
        'coef': res_amp.params[rows_to_display],
        'std err': res_amp.bse[rows_to_display],
        't': res_amp.tvalues[rows_to_display],
        'P>|t|': res_amp.pvalues[rows_to_display],
        '[0.025': res_amp.conf_int().loc[rows_to_display][0],
        '0.975]': res_amp.conf_int().loc[rows_to_display][1]
    }
    
    # DataFrame
    summary_table = pd.DataFrame(results_data)

    # Στρογγυλοποιω την στηλη coef στα 4 δεκαδικα και τις υπολοιπες στα 3 οπως το παραδειγμα
    summary_table['coef'] = summary_table['coef'].round(4)
    cols_to_round_3 = ['std err', 't', 'P>|t|', '[0.025', '0.975]']
    summary_table[cols_to_round_3] = summary_table[cols_to_round_3].round(3)

    print("\n")
    display(summary_table)

except KeyError as e:
    print(f"\nΣφάλμα: Η κατηγορία δεν βρέθηκε: {e}")


### Explanation of Results

The plots visualize how image search amplifies age distortion:

1.  **Grouped Predictions:** In the Control condition, the age gap between Male and Female categories is moderate. However, in the Image (Treatment) condition, the gap widens perceptibly.
2.  **Residuals (Bias):** The residuals plot isolates the bias. We observe that:
    * For **Females**, the treatment residual is negative (~ -0.9) and significantly lower than the control, indicating that visual cues systematically lower the estimated age of women.
    * For **Males**, the treatment residual is positive (~ +1.0), indicating visual cues increase the estimated age of men.

This confirms that online image search does not just reflect societal biases but actively **amplifies** them, making women appear significantly younger and men older than they would in a text-only context.

## 8. Predictions and Residuals Analysis

Για να απομονώσουμε την επίδραση του φύλου και της συνθήκης από τη δυσκολία της κάθε κατηγορίας ή την κρίση του κάθε υποκειμένου:
1.  Τρέχουμε ένα "βασικό" μοντέλο μόνο με `category` και `subj`.
2.  Υπολογίζουμε τα **υπόλοιπα (residuals)**, δηλαδή τη διαφορά μεταξύ της πραγματικής ηλικίας και αυτής που προβλέπει το βασικό μοντέλο.
3.  Φτιάχνουμε δύο γραφήματα (Bar plots):
    * Ένα με την προβλεπόμενη ηλικία ανά ομάδα.
    * Ένα με τα υπόλοιπα (residuals) ανά ομάδα.

Τα residuals μας δείχνουν την "καθαρή" επίδραση της μεροληψίας (bias), αφαιρώντας τον θόρυβο.

In [ ]:
warnings.filterwarnings('ignore')

#Μοντέλο και Προβλέψεις
model_base = smf.ols("age ~ category + subj", data=df_exp)
res_base = model_base.fit()

df_exp['pred_age'] = res_base.predict(df_exp)
df_exp['residuals'] = df_exp['age'] - df_exp['pred_age']

x_order = ['Control', 'Image']
hue_order = ['Female', 'Male'] 
custom_colors = {'Female': '#F4D03F', 'Male': '#1f77b4'} 

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Age Predictions
sns.pointplot(
    ax=axes[0],
    x='condition',
    y='pred_age',
    hue='gender',
    data=df_exp,
    order=x_order,
    hue_order=hue_order,
    palette=custom_colors,
    dodge=0.2,
    capsize=0.1,
    errorbar='ci',
    
    linestyle='-',
    err_kws={'linewidth': 1.2},
    linewidth=1.2,
    markersize=6,
    markers=['o', 'o']
)

axes[0].set_ylabel('predicted')
axes[0].set_xlabel('Condition')
axes[0].grid(False)
# Υπόμνημα
axes[0].legend(
    loc='center right',
    frameon=True,
    facecolor='white',
    framealpha=1
)

# Residuals
sns.pointplot(
    ax=axes[1],
    x='condition',
    y='residuals',
    hue='gender',
    data=df_exp,
    order=x_order,
    hue_order=hue_order,
    palette=custom_colors,
    dodge=0.2,
    capsize=0.1,
    errorbar='ci',
    linestyle='-',
    err_kws={'linewidth': 1.2},
    linewidth=1.2,
    markersize=6,
    markers=['o', 'o']
)

axes[1].set_ylabel('residuals')
axes[1].set_xlabel('Condition')
axes[1].grid(False)

# Υπόμνημα
axes[1].legend(
    title='Gender',
    loc='center right',
    frameon=True,
    facecolor='white',
    framealpha=1
)

plt.tight_layout()
plt.show()

## 9. ANOVA Models

Τέλος, εκτελούμε ανάλυση διακύμανσης (ANOVA) για να ελέγξουμε τη στατιστική σημαντικότητα των παραγόντων.
Χρησιμοποιούμε **Sum coding** (αντί για Treatment coding) και Type 2 ANOVA.

Τρέχουμε δύο μοντέλα:
1.  `age ~ condition * gender`
2.  `age ~ condition * gender + category + subj`

In [ ]:
# ANOVA 1
model_anova1 = smf.ols("age ~ C(condition, Sum) * C(gender, Sum)", data=df_exp)
anova1_res = anova_lm(model_anova1.fit(), typ=2)
print("--- ANOVA Model 1 ---")
display(anova1_res)

# ANOVA 2
model_anova2 = smf.ols("age ~ C(condition, Sum) * C(gender, Sum) + category + subj", data=df_exp)
anova2_res = anova_lm(model_anova2.fit(), typ=2)
print("\n--- ANOVA Model 2 ---")
display(anova2_res)

### ANOVA Interpretation

We performed two ANOVA tests to validate the findings:

* **Model 1 (Basic):** The interaction term `condition:gender` has a p-value of **0.055**, which is marginally above the standard significance threshold (0.05). This suggests that without controlling for other factors, the noise in the data makes the effect harder to detect.
* **Model 2 (Adjusted):** However, when controlling for `category` and `subj` (accounting for job difficulty and participant differences), the interaction `condition:gender` becomes **highly statistically significant** ($p \approx 0.007$).

**Conclusion:** This demonstrates that the amplification effect is robust. Once we account for the variation introduced by different job types and individual participants (Model 2), we statistically confirm that using Google Images significantly alters how gender influences age perception.